# 01_hypothesis_validation

### <b> Adrián Vazquez </b>

---

1. <b> Hipótesis 1: Leader realmente lidera: </b>

Para cada par: 

$return_{leader}(t)  →  return_{follower}(t+1, t+2, t+3...) $ 

Pregunta: Cuando el leader se mueve, ¿el follower se mueve después?

2. <b> Hipótesis 2: La correlación 60 días sirve </b>

Compara: Top correlation pairs vs random same-sector pairs

Pregunta: ¿Los pares top 50 tienen mejor relación lead-lag que pares aleatorios?

3. <b> Hipótesis 3: Market cap como leader tiene sentido.  </b>

Compara dos versiones: 

  -  Leader = mayor market cap
  - Leader = ticker que estadísticamente lidera más

Pregunta: ¿Market cap predice liderazgo o sólo es una regla intuitiva?

4. <b> Hipótesis 4: Los edge factors tienen sentido. </b>

aun no optimizar. Solo medir frecuencia

Pregunta:
 - ¿Cuántas señales genera?
 - ¿Son demasiado raras?
 - ¿O demasiado frecuentes?

5. <b> Hipótesis 5: La señal tiene drift posterior.</b>

Después de señal: follower forward return 5m, 15m, 30m, 60m

Pregunta: 
 - Después de la señal, ¿el follower sube más que en condiciones normales? <- Esta es la prueba más importante antes del backtest

 <b> Output que hay que buscar </b>

 | Hipótesis        |                        Métrica | Resultado          | Decisión              |
| ---------------- | -----------------------------: | ------------------ | --------------------- |
| Leader lidera    |                  Corr lead-lag | positiva / nula    | seguir / ajustar      |
| Top corr aporta  |           diferencia vs random | positiva / nula    | mantener / cambiar    |
| Market cap sirve | hit rate vs leader estadístico | mejor / peor       | mantener / reemplazar |
| Edge factors     |                señales por día | razonable / escaso | ajustar               |
| Drift posterior  |                 forward return | positivo / nulo    | pasar a backtest      |


In [13]:
import pandas as pd
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
from itertools import combinations
sys.path.append(str(Path("..")))
from src.ingest.open_parquet import abrir_parquet_data_polar



In [ ]:
universe_df = pd.read_csv("../data/universe_metadata.csv")
universe_df.head(3)

,symbol,market_cap,sector,company_name
0,AAPL,4.059188e+12,ELECTRONIC COMPUTERS,Apple Inc.
1,ABNB,7.555724e+10,SERVICES-TO DWELLINGS & OTHER BUILDINGS,"Airbnb, Inc. Class A Common Stock"
2,ADBE,1.148210e+11,SERVICES-PREPACKAGED SOFTWARE,Adobe Inc.
3,ADI,1.565761e+11,SEMICONDUCTORS & RELATED DEVICES,"Analog Devices, Inc."
4,ADP,9.448855e+10,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,Automatic Data Processing
...,...,...,...,...
90,WDAY,4.474945e+10,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,"Workday, Inc. Class A Common Stock"
91,WDC,9.134021e+10,COMPUTER STORAGE DEVICES,Western Digital Corp.
92,WMT,1.020181e+12,RETAIL-VARIETY STORES,Walmart Inc. Common Stock
93,XEL,4.507533e+10,ELECTRIC & OTHER SERVICES COMBINED,"Xcel Energy, Inc."


1. Candidate Pair Generation
2. Rolling 60-Day Correlation Ranking
3. Top-50 Pair Selection


In [14]:
#  candidate pair generation 
pair_records = []

for sector, group in universe_df.groupby("sector"):
    symbols = sorted(group["symbol"].unique())

    for stock_a, stock_b in combinations(symbols, 2):
        pair_records.append({
            "stock_a": stock_a,
            "stock_b": stock_b,
            "sector": sector
        })

pairs_df = pd.DataFrame(pair_records)

pairs_df.head()

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES


In [15]:
pairs_df["sector"].value_counts()

sector
SEMICONDUCTORS & RELATED DEVICES                        78
SERVICES-PREPACKAGED SOFTWARE                           78
SERVICES-BUSINESS SERVICES, NEC                         10
PHARMACEUTICAL PREPARATIONS                              6
BEVERAGES                                                3
SERVICES-COMPUTER PROCESSING & DATA PREPARATION          3
SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING, ETC.     3
BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)          1
CABLE & OTHER PAY TELEVISION SERVICES                    1
COMPUTER PERIPHERAL EQUIPMENT, NEC                       1
COMPUTER STORAGE DEVICES                                 1
ELECTRIC & OTHER SERVICES COMBINED                       1
MOTOR VEHICLES & PASSENGER CAR BODIES                    1
RETAIL-CATALOG & MAIL-ORDER HOUSES                       1
RETAIL-VARIETY STORES                                    1
SERVICES-COMPUTER PROGRAMMING SERVICES                   1
Name: count, dtype: int64

In [16]:
pairs_df

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES
...,...,...,...
185,SHOP,TEAM,SERVICES-PREPACKAGED SOFTWARE
186,SHOP,TTWO,SERVICES-PREPACKAGED SOFTWARE
187,SNPS,TEAM,SERVICES-PREPACKAGED SOFTWARE
188,SNPS,TTWO,SERVICES-PREPACKAGED SOFTWARE


In [ ]:
pairs_df.to_csv("../data/candidate_pairs.csv", index=False)


In [ ]:
pairs_df